## Imports

In [21]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

In [3]:
"""
Three publication-ready PDF figures from the V7 fit.
Samples are ordered by film stress, most compressive (−) → most tensile (+).
"""
from matplotlib.backends.backend_pdf import PdfPages

# ──────────────────────────────────────────────────────────────────────
#  Helpers
# ──────────────────────────────────────────────────────────────────────

def _samples_by_stress(fit):
    """Return sample list ordered by stress (ascending: most negative first)."""
    smps = []
    for smp in fit['flat']['SAMPLE_LIST']:
        s = next(rc['stress'] for rc in fit['catalog'] if rc['sample'] == smp)
        smps.append((smp, s))
    smps.sort(key=lambda kv: kv[1])
    return [smp for smp, _ in smps]


def _t_alpha_funcs(sample, fit, T_ref=None):
    """Build invertible (T → α(T)) and (α → T) functions for the secondary
       x-axis. Uses the analytic α(T) = α₀ σ₂(T_ref)/σ₂(T) form.
       Returns (forward, inverse, T_grid, alpha_grid) — falls back to
       interpolating the grid if the closed form is too noisy."""
    if T_ref is None:
        T_ref = T_BASE_POWER
    T_grid = np.linspace(50, max(T_LOSS_MAX, 1100), 500)
    a_grid = alpha_of_T(T_grid, sample, fit, T_ref=T_ref)

    # Enforce strict monotonicity (in case of numerical wiggle)
    a_grid = np.maximum.accumulate(a_grid)

    def fwd(T):
        T = np.asarray(T, dtype=float)
        return np.interp(T, T_grid, a_grid)
    def inv(a):
        a = np.asarray(a, dtype=float)
        return np.interp(a, a_grid, T_grid)
    return fwd, inv, T_grid, a_grid


# ──────────────────────────────────────────────────────────────────────
#  PDF 1 — Qi(T) and δf(T) vs temperature, 2×5 panels, α(T) on top axis
# ──────────────────────────────────────────────────────────────────────

def make_pdf1_T_vs_stress(fit, out_path):
    samples = _samples_by_stress(fit)
    fig, axes = plt.subplots(2, 5, figsize=(22, 8), squeeze=False)
    color_cycle = plt.rcParams['axes.prop_cycle'].by_key()['color']
    T_mod = np.linspace(80, T_LOSS_MAX, 400)
    T_mod_F = np.linspace(80, T_FREQ_MAX, 400)

    for col, smp in enumerate(samples):
        s_val = next(rc['stress'] for rc in fit['catalog'] if rc['sample']==smp)
        ak_smp = fit['ak_per_sample'][smp]
        f2_smp = fit['f2_per_sample'][smp]

        ax_top = axes[0, col]
        ax_bot = axes[1, col]

        rr = [(j, rc) for j, rc in enumerate(fit['catalog'])
              if rc['sample'] == smp and rc['has_T']]

        # ── Upper row: Qi(T) ─────────────────────────────────────────
        for k, (j, rc) in enumerate(rr):
            mask = fit['flat']['L_j'] == j
            T_d  = fit['flat']['L_T'][mask]
            Qi_d = 1.0/fit['flat']['L_loss'][mask]
            err_d = fit['flat']['L_lerr'][mask] / fit['flat']['L_loss'][mask]**2
            c = color_cycle[k % len(color_cycle)]
            ax_top.errorbar(T_d, Qi_d, yerr=err_d, fmt='o', ms=4, alpha=0.7,
                            color=c, label=rc['label'])
            ax_top.plot(T_mod, model_Qi_T_curve(T_mod, j, fit),
                        '-', color=c, lw=1.4, alpha=0.9)

        ax_top.set_yscale('log')
        ax_top.grid(alpha=0.3, which='both')
        ax_top.set_title(f"{smp}  σ={s_val:+.1f} MPa\n"
                         rf"$\alpha_k$={ak_smp:.3f}, $f_2$={f2_smp:.2e}",
                         fontsize=10)
        if col == 0:
            ax_top.set_ylabel(r'$Q_i$')
        ax_top.set_xlabel('')
        ax_top.set_xticklabels([])
        ax_top.legend(fontsize=7, loc='best')

        # α(T) secondary axis on the upper panel
        fwd, inv, _, _ = _t_alpha_funcs(smp, fit)
        sec = ax_top.secondary_xaxis('top', functions=(fwd, inv))
        # Set 4 evenly-spaced ticks across the T range to prevent
        # overlapping labels when α barely varies with T
        T_tick = np.linspace(100, T_LOSS_MAX, 4)
        a_tick = fwd(T_tick)
        sec.set_xticks(a_tick)
        a_span = a_tick.max() - a_tick.min()
        # When α range is tiny (Δα/α < 1%) show 4 decimals; else 3
        rel = a_span / max(abs(a_tick.mean()), 1e-30)
        fmt = '{:.4f}' if rel < 0.01 else '{:.3f}'
        sec.set_xticklabels([fmt.format(a) for a in a_tick])
        sec.set_xlabel(r'$\alpha(T)$', fontsize=9)
        sec.tick_params(labelsize=8)

        # ── Lower row: δf(T) ─────────────────────────────────────────
        for k, (j, rc) in enumerate(rr):
            mask = fit['flat']['F_j'] == j
            T_d   = fit['flat']['F_T'][mask]
            dff_d = fit['flat']['F_dff'][mask]
            err_d = fit['flat']['F_dferr'][mask]
            c = color_cycle[k % len(color_cycle)]
            ax_bot.errorbar(T_d, dff_d*1e6, yerr=err_d*1e6, fmt='o', ms=4,
                            alpha=0.7, color=c, label=rc['label'])
            ax_bot.plot(T_mod_F, model_df_T_curve(T_mod_F, j, fit)*1e6,
                        '-', color=c, lw=1.4, alpha=0.9)

        ax_bot.axhline(0, color='k', lw=0.5, alpha=0.3)
        ax_bot.grid(alpha=0.3)
        ax_bot.set_xlabel('T (mK)')
        if col == 0:
            ax_bot.set_ylabel(r'$\delta f / f_0$  (ppm)')

    fig.suptitle(r'$Q_i(T)$ and $\delta f/f_0(T)$ vs stress  (samples in order of increasing $\sigma$)',
                 fontsize=13)
    fig.tight_layout(rect=[0, 0, 1, 0.96])
    with PdfPages(out_path) as pdf:
        pdf.savefig(fig, bbox_inches='tight')
    plt.close(fig)
    return out_path


# ──────────────────────────────────────────────────────────────────────
#  PDF 2 — Qi(n) and δf(n) vs ⟨n⟩, 2×5 panels (no α(T) axis)
# ──────────────────────────────────────────────────────────────────────

def make_pdf2_n_vs_stress(fit, out_path):
    samples = _samples_by_stress(fit)
    fig, axes = plt.subplots(2, 5, figsize=(22, 8), squeeze=False)
    color_cycle = plt.rcParams['axes.prop_cycle'].by_key()['color']
    n_mod = np.logspace(np.log10(N_PHOTON_MIN), np.log10(N_PHOTON_MAX), 300)

    for col, smp in enumerate(samples):
        s_val = next(rc['stress'] for rc in fit['catalog'] if rc['sample']==smp)
        n_c_smp = fit['n_c_per_sample'][smp]
        dT_smp  = fit['dT_per_sample'][smp]

        ax_top = axes[0, col]
        ax_bot = axes[1, col]

        # ── Upper row: Qi(n) ─────────────────────────────────────────
        rr_P = [(j, rc) for j, rc in enumerate(fit['catalog'])
                if rc['sample'] == smp and rc['has_P']]
        for k, (j, rc) in enumerate(rr_P):
            mask = fit['flat']['P_j'] == j
            n_d  = fit['flat']['P_n'][mask]
            Qi_d = 1.0/fit['flat']['P_loss'][mask]
            err_d = fit['flat']['P_lerr'][mask] / fit['flat']['P_loss'][mask]**2
            c = color_cycle[k % len(color_cycle)]
            ax_top.errorbar(n_d, Qi_d, yerr=err_d, fmt='o', ms=4, alpha=0.7,
                            color=c, label=rc['label'])
            ax_top.plot(n_mod, model_Qi_n_curve(n_mod, j, fit),
                        '-', color=c, lw=1.4, alpha=0.9)

        ax_top.axvline(n_c_smp, color='gray', ls=':', alpha=0.5, lw=1)
        ax_top.set_xscale('log'); ax_top.set_yscale('log')
        ax_top.grid(alpha=0.3, which='both')
        ax_top.set_title(f"{smp}  σ={s_val:+.1f} MPa\n"
                         rf"$n_c$={n_c_smp:.2g}, $\delta_{{TLS,0}}$={dT_smp:.2e}",
                         fontsize=10)
        if col == 0:
            ax_top.set_ylabel(r'$Q_i$')
        ax_top.set_xlabel('')
        ax_top.set_xticklabels([])
        ax_top.legend(fontsize=7, loc='best')

        # ── Lower row: δf(n) ─────────────────────────────────────────
        rr_PF = [(j, rc) for j, rc in enumerate(fit['catalog'])
                 if rc['sample'] == smp and rc['has_PF']]
        for k, (j, rc) in enumerate(rr_PF):
            mask = fit['flat']['PF_j'] == j
            n_d   = fit['flat']['PF_n'][mask]
            dff_d = fit['flat']['PF_dff'][mask]
            err_d = fit['flat']['PF_dferr'][mask]
            c = color_cycle[k % len(color_cycle)]
            ax_bot.errorbar(n_d, dff_d*1e6, yerr=err_d*1e6, fmt='o', ms=4,
                            alpha=0.7, color=c, label=rc['label'])
            ax_bot.plot(n_mod, model_df_n_curve(n_mod, j, fit)*1e6,
                        '-', color=c, lw=1.4, alpha=0.9)

        ax_bot.axvline(n_c_smp, color='gray', ls=':', alpha=0.5, lw=1)
        ax_bot.axhline(0, color='k', lw=0.5, alpha=0.3)
        ax_bot.set_xscale('log')
        ax_bot.grid(alpha=0.3, which='both')
        ax_bot.set_xlabel(r'$\langle n \rangle$')
        if col == 0:
            ax_bot.set_ylabel(r'$\delta f / f_0$  (ppm)')

    fig.suptitle(r'$Q_i(\langle n\rangle)$ and $\delta f/f_0(\langle n\rangle)$ at base $T$  '
                 r'(samples in order of increasing $\sigma$)', fontsize=13)
    fig.tight_layout(rect=[0, 0, 1, 0.96])
    with PdfPages(out_path) as pdf:
        pdf.savefig(fig, bbox_inches='tight')
    plt.close(fig)
    return out_path


# ──────────────────────────────────────────────────────────────────────
#  PDF 3 — α vs stress: base T and isotherm view
# ──────────────────────────────────────────────────────────────────────

def make_pdf3_alpha_vs_stress(fit, out_path, isotherms_mK=(100.0, 500.0, 1000.0)):
    samples = _samples_by_stress(fit)
    stresses    = np.array([next(rc['stress'] for rc in fit['catalog']
                                  if rc['sample']==s) for s in samples])
    stress_errs = np.array([next(rc['stress_err'] for rc in fit['catalog']
                                  if rc['sample']==s) for s in samples])

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

    # Panel 1: α at base T only
    alpha_base = np.array([fit['ak_per_sample'][s] for s in samples])
    ax1.errorbar(stresses, alpha_base, xerr=stress_errs, fmt='o', ms=8,
                 color='C0', capsize=4, lw=1.2)
    for s_val, a_val, smp in zip(stresses, alpha_base, samples):
        ax1.annotate(smp.replace('Sample ', ''),
                     (s_val, a_val), textcoords='offset points',
                     xytext=(7, 5), fontsize=9, color='C0')
    ax1.axvline(0, color='k', lw=0.5, alpha=0.4)
    ax1.set_xlabel(r'Film stress  $\sigma$  (MPa)')
    ax1.set_ylabel(r'$\alpha_k$  at base $T$ ({:.0f} mK)'.format(T_BASE_POWER))
    ax1.set_title(r'$\alpha_k$ vs film stress at base $T$')
    ax1.grid(alpha=0.3)

    # Panel 2: α isotherms vs σ
    cmap = plt.cm.viridis
    colors = [cmap(x) for x in np.linspace(0.15, 0.85, len(isotherms_mK))]
    markers = ['o', 's', '^', 'D', 'v']
    for i, (T_iso, c) in enumerate(zip(isotherms_mK, colors)):
        alpha_iso = np.array([alpha_of_T(np.array([T_iso]), s, fit)[0]
                              for s in samples])
        if T_iso < 200: lbl = f'base T ({T_iso:.0f} mK)'
        elif T_iso < 1000: lbl = f'{T_iso:.0f} mK'
        else: lbl = f'{T_iso/1000:.1f} K'
        ax2.errorbar(stresses, alpha_iso, xerr=stress_errs,
                     fmt=markers[i % len(markers)] + '-', ms=8 - 2*i,
                     mfc='white' if i > 0 else c, mec=c,
                     color=c, capsize=3, lw=1.4, alpha=0.9, label=lbl)
    ax2.axvline(0, color='k', lw=0.5, alpha=0.4)
    ax2.set_xlabel(r'Film stress  $\sigma$  (MPa)')
    ax2.set_ylabel(r'$\alpha(T)$')
    ax2.set_title(r'$\alpha(T)$ isotherms vs film stress')
    ax2.grid(alpha=0.3)
    ax2.legend(loc='best')

    fig.tight_layout()
    with PdfPages(out_path) as pdf:
        pdf.savefig(fig, bbox_inches='tight')
    plt.close(fig)
    return out_path

In [8]:
import os
os.makedirs("pdfs", exist_ok=True)

p1 = make_pdf1_T_vs_stress(fit_v7,     "pdfs/fig1_Qi_df_vs_T.pdf")
p2 = make_pdf2_n_vs_stress(fit_v7,     "pdfs/fig2_Qi_df_vs_n.pdf")
p3 = make_pdf3_alpha_vs_stress(fit_v7, "pdfs/fig3_alpha_vs_stress.pdf")

for p in (p1, p2, p3):
    print(f"  {p}: {os.path.getsize(p)/1024:.1f} KB")

NameError: name 'fit_v7' is not defined

---
## Publication Figure Builder

Cells below are self-contained figure generators for paper-ready output.  
**Workflow:**
1. Run **Cell 1** (data import) and **Cell 2** (formatting setup) once per session.
2. Run **Cell 3** (image loader helper) if any figure panel uses a saved PDF/SVG.
3. Each of **Figure Cells 1–5** is independent — run any subset, in any order.
4. Each figure cell saves a PDF to `./paper_figures/` *and* displays inline.

> **Collaborator note:** Edit the `# ── CONFIG` block at the top of each figure cell  
> to change panel layout, labels, which resonators are plotted, axis limits, etc.  
> Everything below `# ── END CONFIG` is rendering machinery — change with care.

In [22]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  CELL 1 — Data import                                               ║
# ║  Imports curated_data.py (must live next to this notebook) and       ║
# ║  exposes `data_dict` for all figure cells.                           ║
# ║  If the notebook already loaded stress_resonators via pickle above,  ║
# ║  that dict is also available under its original name.                ║
# ╚══════════════════════════════════════════════════════════════════════╝

import importlib, sys
from pathlib import Path

# ── Locate curated_data.py relative to this notebook ──────────────────────
NOTEBOOK_DIR = Path().resolve()          # works when running from notebook dir
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

import curated_data                       # imports the module
importlib.reload(curated_data)           # re-import picks up any edits on disk

data_dict = curated_data.data_dict       # the main dictionary used by figure cells

# ── Quick sanity check ────────────────────────────────────────────────────
print(f"data_dict loaded with {len(data_dict)} samples:")
for sample, chip in data_dict.items():
    res_keys = [k for k, v in chip.items() if isinstance(v, dict)]
    stress   = chip.get('stress', 'N/A')
    print(f"  {sample:<12} σ = {stress:>+8}  |  {len(res_keys)} resonators: {res_keys}")


data_dict loaded with 6 samples:
  Sample A     σ =     -734  |  4 resonators: ['Res1_NoTemp', 'Res2', 'Res3', 'Res4']
  Sample B     σ =     -535  |  6 resonators: ['Res1', 'Res2', 'Res3', 'Res4', 'Res5_NoTemp', 'Res6_NoTemp']
  Sample C     σ =     -368  |  3 resonators: ['Res1', 'Res2', 'Res3']
  Sample D     σ =      -41  |  8 resonators: ['Res1', 'Res2', 'Res3', 'Res4', 'Res5_NoTemp', 'Res6_NoTemp', 'Res7_NoTemp', 'Res8_NoTemp']
  Sample E     σ =    +62.5  |  3 resonators: ['Res1', 'Res2', 'Res3']
  Sample N     σ =     -368  |  2 resonators: ['Res1', 'Res2']


In [23]:
# ── External figure viewer (non-inline backend) ──────────────────────────
# Switch to an interactive backend so figures open in a separate window
# at full resolution instead of being embedded in the notebook output.
# Change the backend name if your system uses a different toolkit:
#   TkAgg  — Tkinter (usually available everywhere)
#   Qt5Agg — Qt5 (requires PyQt5 or PySide2)
#   MacOSX — macOS native (fastest on Mac)
import matplotlib
try:
    matplotlib.use('Qt5Agg')
except Exception:
    try:
        matplotlib.use('TkAgg')
    except Exception:
        pass   # fall back to whatever backend is already active

# ╔══════════════════════════════════════════════════════════════════════╗
# ║  CELL 2 — Unified matplotlib formatting                             ║
# ║  Run once per session.                                               ║
# ║  All style is controlled exclusively through FMT below.             ║
# ║  There is no separate rcParams block — FMT is the single source.    ║
# ╚══════════════════════════════════════════════════════════════════════╝

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
from matplotlib.backends.backend_pdf import PdfPages
from pathlib import Path
import warnings

# ── FMT — single source of truth for all style ───────────────────────────
# Edit values here to change every figure at once.
# Font sizes, tick sizes, line widths, margins, colors — all live here.
FMT = dict(
    # Fonts
    font_family   = 'sans-serif',
    font_size     = 16,   # base font (axes labels, ticks, legend)
    title_size    = 0,    # panel/figure title size (0 = hidden)

    # Markers and lines
    color_list    = ['red','orange','gold','green','blue','indigo','violet',
                     'cyan','brown','black','lightseagreen','coral','purple',
                     'seagreen','springgreen','aquamarine','mediumslateblue'],
    marker_list   = ['o','s','^','D','v','>','<','p','*','h','H','+','x'],
    marker_size   = 8,
    cap_size      = 8,
    err_lw        = 2,
    line_width    = 2,    # data line width
    axes_lw       = 1.5,  # axes spine width

    # Tick style
    x_tick_size   = 16,
    x_tick_minor_len  = 8,
    x_tick_major_len  = 14,
    x_tick_minor_w    = 2,
    x_tick_major_w    = 3,
    x_pad         = 4,    # pts between tick end and label
    y_tick_size   = 16,
    y_tick_minor_len  = 8,
    y_tick_major_len  = 14,
    y_tick_minor_w    = 2,
    y_tick_major_w    = 3,
    y_pad         = 4,    # pts between tick end and label

    # Labels
    x_label_size  = 16,
    y_label_size  = 16,
    legend_size        = 11,   # font size for the figure legend
    legend_marker_scale = 0.7,  # marker size = marker_size * this

    # Figure margins (fraction of figure)
    left          = 0.1,
    right         = 0.9,
    bottom        = 0.1,
    top           = 0.9,

    # Save / PDF
    figure_dpi    = 150,
    save_dpi      = 300,
    pdf_fonttype  = 42,   # embeds fonts (required by most journals)
)

# Apply the FMT font settings globally so text rendered outside axes
# (e.g. colorbars, suptitle) also respects them.
matplotlib.rcParams['font.family']      = FMT['font_family']
matplotlib.rcParams['font.size']        = FMT['font_size']
matplotlib.rcParams['figure.dpi']       = FMT['figure_dpi']
matplotlib.rcParams['savefig.dpi']      = FMT['save_dpi']
matplotlib.rcParams['savefig.bbox']     = 'tight'
matplotlib.rcParams['lines.linewidth']  = FMT['line_width']
matplotlib.rcParams['axes.linewidth']   = FMT['axes_lw']
matplotlib.rcParams['pdf.fonttype']     = FMT['pdf_fonttype']
matplotlib.rcParams['ps.fonttype']      = FMT['pdf_fonttype']

# ── Per-sample marker and color styles ────────────────────────────────────
# Maps each data_dict key to a (color, marker) tuple.
# Edit these to control exactly how each sample appears in every figure.
# If a sample key is not listed here, a style will be assigned automatically.
SAMPLE_STYLES = {
    'Sample A': ('red',    's'),
    'Sample B': ('indigo', '^'),
    'Sample C': ('orange', 'v'),
    'Sample D': ('blue',   'o'),
    'Sample E': ('cyan',   'D'),
    'Sample N': ('purple', 'p'),
    # Add more samples here as needed:
    # 'Sample X': ('cyan', '*'),
}


def get_sample_style(sample_name, resonator_index=0):
    """Return (color, marker) for a sample, falling back to auto-assignment."""
    if sample_name in SAMPLE_STYLES:
        return SAMPLE_STYLES[sample_name]
    ci = list(data_dict.keys()).index(sample_name) if sample_name in data_dict else resonator_index
    color  = FMT['color_list'][ci % len(FMT['color_list'])]
    marker = FMT['marker_list'][ci % len(FMT['marker_list'])]
    return color, marker


def make_legend_handles(entries):
    """Build custom legend handles.

    entries : list of sample-name strings OR (label, color, marker) tuples.
      - String  → looks up SAMPLE_STYLES for color/marker and data_dict for
                  the stress value to use as the label ("<stress> MPa").
      - Tuple   → (label, color, marker) used as-is.
    marker=None in a tuple → solid colour patch.
    """
    handles = []
    for entry in entries:
        if isinstance(entry, str):
            # Auto-derive from data_dict stress value and SAMPLE_STYLES
            sample = entry
            color, marker = get_sample_style(sample)
            stress = data_dict.get(sample, {}).get('stress', None)
            if stress is not None:
                label = f'{int(stress):+d} MPa'
            else:
                label = sample
        else:
            label, color, marker = entry

        if marker is None:
            h = mpatches.Patch(color=color, label=label)
        else:
            h = mlines.Line2D(
                [], [],
                color=color,
                marker=marker,
                linestyle='-',
                markersize=FMT['marker_size'] * FMT['legend_marker_scale'],
                label=label,
            )
        handles.append(h)
    return handles


def begin_figure(title='', figsize=(10, 4)):
    """Open a figure + axes with locked physical dimensions, return (fig, ax)."""
    fig, ax = plt.subplots(figsize=figsize)
    plt.subplots_adjust(
        left=FMT['left'], right=FMT['right'],
        bottom=FMT['bottom'], top=FMT['top']
    )
    if title:
        ax.set_title(title, fontsize=FMT['title_size'])
    return fig, ax


def apply_axes_style(ax, xlabel='', ylabel='', xscale='linear', yscale='linear'):
    """Apply tick style and labels driven entirely by FMT."""
    ax.set_xscale(xscale)
    ax.set_yscale(yscale)
    ax.yaxis.set_minor_locator(ticker.AutoMinorLocator())
    ax.xaxis.set_minor_locator(ticker.AutoMinorLocator())
    ax.tick_params(axis='x', which='minor',
                   labelsize=FMT['x_tick_size'],
                   width=FMT['x_tick_minor_w'], length=FMT['x_tick_minor_len'],
                   direction='in', top=True, pad=FMT['x_pad'])
    ax.tick_params(axis='x', which='major',
                   labelsize=FMT['x_tick_size'],
                   width=FMT['x_tick_major_w'], length=FMT['x_tick_major_len'],
                   direction='in', top=True, pad=FMT['x_pad'])
    ax.tick_params(axis='y', which='minor',
                   labelsize=FMT['y_tick_size'],
                   width=FMT['y_tick_minor_w'], length=FMT['y_tick_minor_len'],
                   direction='in', right=True, pad=FMT['y_pad'])
    ax.tick_params(axis='y', which='major',
                   labelsize=FMT['y_tick_size'],
                   width=FMT['y_tick_major_w'], length=FMT['y_tick_major_len'],
                   direction='in', right=True, pad=FMT['y_pad'])
    # Label font sizes pulled from FMT
    if xlabel:
        ax.set_xlabel(xlabel, fontsize=FMT['x_label_size'])
    if ylabel:
        ax.set_ylabel(ylabel, fontsize=FMT['y_label_size'])
    # Spine width
    for spine in ax.spines.values():
        spine.set_linewidth(FMT['axes_lw'])


def _plot_base(ax, x, y, yerr, xerr, label, color, marker, fmt_override):
    """Internal helper: draw one errorbar series onto ax."""
    style = dict(
        fmt        = marker + '-',
        ms         = FMT['marker_size'],
        capsize    = FMT['cap_size'],
        elinewidth = FMT['err_lw'],
        label      = label,
        color      = color,
    )
    style.update(fmt_override or {})
    ax.errorbar(x, y, yerr=yerr, xerr=xerr, **style)


def plot_linear(ax, x, y, yerr=None, xerr=None,
                xlabel='', ylabel='', label='', color='blue', marker='o', **kw):
    _plot_base(ax, x, y, yerr, xerr, label, color, marker, kw)
    apply_axes_style(ax, xlabel, ylabel, xscale='linear', yscale='linear')

def plot_loglog(ax, x, y, yerr=None, xerr=None,
                xlabel='', ylabel='', label='', color='blue', marker='o', **kw):
    _plot_base(ax, x, y, yerr, xerr, label, color, marker, kw)
    apply_axes_style(ax, xlabel, ylabel, xscale='log', yscale='log')

def plot_semilogy(ax, x, y, yerr=None, xerr=None,
                  xlabel='', ylabel='', label='', color='blue', marker='o', **kw):
    _plot_base(ax, x, y, yerr, xerr, label, color, marker, kw)
    apply_axes_style(ax, xlabel, ylabel, xscale='linear', yscale='log')

def plot_semilogx(ax, x, y, yerr=None, xerr=None,
                  xlabel='', ylabel='', label='', color='blue', marker='o', **kw):
    _plot_base(ax, x, y, yerr, xerr, label, color, marker, kw)
    apply_axes_style(ax, xlabel, ylabel, xscale='log', yscale='linear')


def save_figure(fig, filename, out_dir='paper_figures', dpi=None):
    """Save figure as PDF and open in the external viewer.
    Returns the Path of the saved file."""
    dpi = dpi or FMT['save_dpi']
    out = Path(out_dir)
    out.mkdir(exist_ok=True)
    pdf_path = out / (filename if filename.endswith('.pdf') else filename + '.pdf')
    try:
        fig.tight_layout()   # reflow layout before saving
    except Exception:
        pass
    with PdfPages(str(pdf_path)) as pdf:
        pdf.savefig(fig, bbox_inches='tight', dpi=dpi)
    print(f'  Saved → {pdf_path}  ({pdf_path.stat().st_size/1024:.1f} KB)')
    plt.show()   # displays in the external Python figure viewer
    return pdf_path


print('Formatting helpers ready.  FMT is the sole style control — no rcParams block.')
print(f'Legend font size: {FMT["legend_size"]}pt  marker scale: {FMT["legend_marker_scale"]}')
print('Output folder: ./paper_figures/')
print('\nPer-sample styles (SAMPLE_STYLES):')
for k, (c, m) in SAMPLE_STYLES.items():
    print(f'  {k:<12} → color={c:<14} marker={m}')
print('\nUse make_legend_handles([(label, color, marker), ...]) to build custom legends.')


Formatting helpers ready.  FMT is the sole style control — no rcParams block.
Legend font size: 11pt  marker scale: 0.7
Output folder: ./paper_figures/

Per-sample styles (SAMPLE_STYLES):
  Sample A     → color=red            marker=s
  Sample B     → color=indigo         marker=^
  Sample C     → color=orange         marker=v
  Sample D     → color=blue           marker=o
  Sample E     → color=cyan           marker=D
  Sample N     → color=purple         marker=p

Use make_legend_handles([(label, color, marker), ...]) to build custom legends.


In [24]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  CELL 3 — Image / external panel loader                             ║
# ║  Use load_image_panel() to drop a saved PDF or SVG into any         ║
# ║  matplotlib axes object as a panel inside a composite figure.       ║
# ╚══════════════════════════════════════════════════════════════════════╝

import io
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.image as mpimg


def load_image_panel(ax, image_path, title_label=None,
                     interpolation='antialiased', aspect='equal'):
    """
    Render an external image file (PNG, JPEG, PDF rasterised, SVG rasterised)
    into axes `ax`, turning off all tick marks so it looks like a figure panel.

    Parameters
    ----------
    ax           : matplotlib Axes to draw into
    image_path   : str or Path — PNG / JPEG / SVG / PDF
    title_label  : optional panel label, e.g. '(a)' drawn in the top-left corner
    interpolation: passed to imshow
    aspect       : 'equal' (default) or 'auto'

    Notes
    -----
    • PNG / JPEG: loaded directly with matplotlib.
    • SVG        : converted to PNG in memory via cairosvg (pip install cairosvg).
    • PDF        : first page rasterised via pdf2image / poppler
                   (pip install pdf2image  +  apt install poppler-utils).
    """
    p = Path(image_path)
    suffix = p.suffix.lower()

    if suffix in ('.png', '.jpg', '.jpeg'):
        img = mpimg.imread(str(p))

    elif suffix == '.svg':
        try:
            import cairosvg
            png_bytes = cairosvg.svg2png(url=str(p))
            img = mpimg.imread(io.BytesIO(png_bytes))
        except ImportError:
            raise ImportError(
                'cairosvg is required for SVG panels.\n'
                'Install with:  pip install cairosvg'
            )

    elif suffix == '.pdf':
        try:
            from pdf2image import convert_from_path
            pages = convert_from_path(str(p), dpi=200, first_page=1, last_page=1)
            img = np.array(pages[0])
        except ImportError:
            raise ImportError(
                'pdf2image + poppler are required for PDF panels.\n'
                'Install with:  pip install pdf2image\n'
                '               sudo apt install poppler-utils'
            )
    else:
        raise ValueError(f'Unsupported image format: {suffix!r}')

    ax.imshow(img, interpolation=interpolation, aspect=aspect)
    ax.axis('off')

    if title_label is not None:
        ax.text(0.02, 0.97, title_label,
                transform=ax.transAxes,
                fontsize=FMT['title_size'], fontweight='bold',
                va='top', ha='left')


print('load_image_panel() ready.')
print('Supported formats: PNG, JPEG, SVG (needs cairosvg), PDF (needs pdf2image + poppler)')


load_image_panel() ready.
Supported formats: PNG, JPEG, SVG (needs cairosvg), PDF (needs pdf2image + poppler)


### Figure Cell 1 (Old Figure 1)
*Qi vs photon number*  
Edit `FIG1_*` variables in the CONFIG block to customise.

In [29]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  FIGURE CELL 1 — Figure 1 Title                                       ║
# ╚══════════════════════════════════════════════════════════════════════╝
# Requires: Cell 1 (data_dict) and Cell 2 (formatting helpers).
# Edit the CONFIG block below; do not change code below '# ── END CONFIG'.

# ── CONFIG ────────────────────────────────────────────────────────────────

FIG1_TITLE    = 'Figure 1 Title'
FIG1_LABEL    = '(a)'        # panel label drawn in top-left corner of panel 0
FIG1_FILENAME = 'fig1'               # saved to ./paper_figures/

# Figure dimensions in inches (width, height). PHYSICAL size of the exported PDF.
FIG1_WIDTH  = 10   # inches
FIG1_HEIGHT = 4    # inches

# Grid layout: set rows × cols explicitly.
# Unused trailing slots are hidden automatically.
FIG1_ROWS = 1   # number of subplot rows
FIG1_COLS = 3   # number of subplot columns

# ── Figure-level legend ───────────────────────────────────────────────────
# One legend for the whole figure, drawn outside the panels.
# Each entry is a tuple: (label_string, color, marker_shape)
#
# marker_shape uses standard matplotlib marker codes, e.g.:
#   'o' circle  's' square  '^' triangle-up  'D' diamond
#   'v' triangle-down  '>' triangle-right  'p' pentagon  '*' star
#
# The legend is placed to the right of the rightmost panel by default.
# Set FIG1_LEGEND = [] to suppress the legend entirely.

# List sample names to include in the legend (order = display order).
# Each name is looked up in SAMPLE_STYLES for color/marker and in data_dict
# for its stress value, which becomes the label ("<stress> MPa").
# You can also mix in manual (label, color, marker) tuples if needed.
FIG1_LEGEND = [
    'Sample A',
    'Sample B',
    'Sample C',
    'Sample D',
    'Sample E',
    'Sample N',
]
# Legend anchor position in figure-fraction coordinates (0–1).
# (1.0, 0.5) = right edge, vertically centred.  Increase X beyond 1 to
# push further right if panels are wide; decrease to move left into whitespace.
FIG1_LEGEND_X = 0.91  # horizontal anchor (figure fraction); sits in the right-margin gap
FIG1_LEGEND_Y = 0.5   # vertical anchor   (figure fraction)

# ── Per-panel configuration ───────────────────────────────────────────────
# FIG1_PANELS is a list of dicts, one per panel (left-to-right, top-to-bottom).
# Panels beyond ROWS × COLS are ignored. Unfilled grid slots are hidden.
#
# Required keys per panel dict:
#   'samples'            : list of sample names from data_dict to plot in this panel.
#                          Use None or [] to leave the panel blank.
#   'resonators'         : list of resonator keys to include, e.g. ['Res1', 'Res2'].
#                          Use None to plot ALL resonators found in each sample.
#   'exclude_resonators' : list of resonator keys to skip, e.g. ['Res1'].
#                          Applied after 'resonators'; use None to skip nothing.
#                          Per-sample exclusions: dict keyed by sample name, e.g.
#                          {'Sample A': ['Res1'], 'Sample B': ['Res3']}
#   'x_key'              : data_dict key for the x axis, e.g. 'Num_Photon' or 'Temp'.
#   'y_key'              : data_dict key for the y axis, e.g. 'Qi_Power' or 'Qi_Temp'.
#   'ye_key'             : data_dict key for y error bars (set to None to suppress).
#
# Optional keys per panel dict:
#   'xlabel'     : x-axis label string  (default: '')
#   'ylabel'     : y-axis label string  (default: '')
#   'xscale'     : 'linear' or 'log'   (default: 'linear')
#   'yscale'     : 'linear' or 'log'   (default: 'linear')
#   'xlim'       : (lo, hi) or None    (default: None = auto)
#   'ylim'       : (lo, hi) or None    (default: None = auto)
#   'title'      : panel title string  (default: None = no title)
#
# Common x/y key names:
#   Qi_Temp, Qi_Power, Num_Photon, Temp, Freq,
#   Q_Temp, Qc_Temp, Freq_err_Temp, Power_Freq, Power_Freq_err

FIG1_PANELS = [
    # Panel 0 — image panel (loads ChipBoundGray.png from the notebook directory)
    dict(
        image_path          = 'ChipBoundGray.png',   # path relative to notebook directory
        title               = '',                    # optional panel title
    ),
    # Panel 1
    dict(
        samples             = ['Sample B'],
        resonators          = None,
        exclude_resonators  = None,
        x_key               = 'Num_Photon',
        y_key               = 'Qi_Power',
        ye_key              = 'Qi_Power_err',
        xlabel              = r'$\langle n \rangle$ (photons)',
        ylabel              = r'$Q_i$ $(10^6)$',
        xscale              = 'log',
        yscale              = 'log',
        xlim                = None,
        ylim                = None,
        title               = 'Sample B',
    ),
    # Panel 2
    dict(
        samples             = ['Sample C'],
        resonators          = None,
        exclude_resonators  = None,
        x_key               = 'Num_Photon',
        y_key               = 'Qi_Power',
        ye_key              = 'Qi_Power_err',
        xlabel              = r'$\langle n \rangle$ (photons)',
        ylabel              = r'$Q_i$ $(10^6)$',
        xscale              = 'log',
        yscale              = 'log',
        xlim                = None,
        ylim                = None,
        title               = 'Sample C',
    ),
    # Panel 3
    dict(
        samples             = ['Sample D'],
        resonators          = None,
        exclude_resonators  = None,
        x_key               = 'Num_Photon',
        y_key               = 'Qi_Power',
        ye_key              = 'Qi_Power_err',
        xlabel              = r'$\langle n \rangle$ (photons)',
        ylabel              = r'$Q_i$ $(10^6)$',
        xscale              = 'log',
        yscale              = 'log',
        xlim                = None,
        ylim                = None,
        title               = 'Sample D',
    ),
    # Panel 4
    dict(
        samples             = ['Sample E'],
        resonators          = None,
        exclude_resonators  = None,
        x_key               = 'Num_Photon',
        y_key               = 'Qi_Power',
        ye_key              = 'Qi_Power_err',
        xlabel              = r'$\langle n \rangle$ (photons)',
        ylabel              = r'$Q_i$ $(10^6)$',
        xscale              = 'log',
        yscale              = 'log',
        xlim                = None,
        ylim                = None,
        title               = 'Sample E',
    ),
    # Panel 5
    dict(
        samples             = ['Sample N'],
        resonators          = None,
        exclude_resonators  = None,
        x_key               = 'Num_Photon',
        y_key               = 'Qi_Power',
        ye_key              = 'Qi_Power_err',
        xlabel              = r'$\langle n \rangle$ (photons)',
        ylabel              = r'$Q_i$ $(10^6)$',
        xscale              = 'log',
        yscale              = 'log',
        xlim                = None,
        ylim                = None,
        title               = 'Sample N',
    ),
    # Panel 6 — overlay multiple samples on one panel
    dict(
        samples             = ['Sample A', 'Sample B'],
        resonators          = ['Res1', 'Res2'],
        exclude_resonators  = None,
        x_key               = 'Num_Photon',
        y_key               = 'Qi_Power',
        ye_key              = 'Qi_Power_err',
        xlabel              = r'$\langle n \rangle$ (photons)',
        ylabel              = r'$Q_i$ $(10^6)$',
        xscale              = 'log',
        yscale              = 'log',
        xlim                = None,
        ylim                = None,
        title               = 'A vs B (Res1, Res2)',
    ),
    # Panel 7
    dict(
        samples             = ['Sample C', 'Sample D'],
        resonators          = None,
        exclude_resonators  = None,
        x_key               = 'Temp',
        y_key               = 'Qi_Temp',
        ye_key              = 'Qi_Temp_err',
        xlabel              = 'T (mK)',
        ylabel              = r'$Q_i$ $(10^6)$',
        xscale              = 'linear',
        yscale              = 'log',
        xlim                = None,
        ylim                = None,
        title               = 'Loss vs T',
    ),
]

# ── END CONFIG ────────────────────────────────────────────────────────────

# ── Build figure ──────────────────────────────────────────────────────────
_n_slots1 = FIG1_ROWS * FIG1_COLS
_n_panels1 = min(len(FIG1_PANELS), _n_slots1)

_fig1_size = (FIG1_WIDTH, FIG1_HEIGHT)
if FIG1_ROWS == 1 and FIG1_COLS == 1:
    fig1, _ax_raw1 = plt.subplots(figsize=_fig1_size,
                                      layout='constrained')
    _axes_flat1 = np.array([_ax_raw1])
else:
    fig1, _ax_raw1 = plt.subplots(FIG1_ROWS, FIG1_COLS, figsize=_fig1_size,
                                      squeeze=False, layout='constrained')
    _axes_flat1 = np.array(_ax_raw1).flatten()

# layout='constrained' (set above) manages margins automatically.
# subplots_adjust is not used — it conflicts with constrained_layout.
# To add extra padding around the figure, edit FMT['left'/'right'/'bottom'/'top']
# then call fig1.set_constrained_layout_pads(w_pad=..., h_pad=...) after this block.

# Hide unused grid slots
for _k1 in range(_n_panels1, len(_axes_flat1)):
    _axes_flat1[_k1].axis('off')

# ── Plotting loop (no per-panel legend) ───────────────────────────────────
for _pi1, _pcfg1 in enumerate(FIG1_PANELS[:_n_panels1]):
    _ax1 = _axes_flat1[_pi1]

    # Image panel — skip the data loop and render the image instead
    if 'image_path' in _pcfg1:
        load_image_panel(_ax1, _pcfg1['image_path'],
                         title_label=_pcfg1.get('title') or None)
        continue

    _samples1    = _pcfg1.get('samples') or []
    _res_filter1 = _pcfg1.get('resonators')
    _xk1         = _pcfg1['x_key']
    _yk1         = _pcfg1['y_key']
    _yek1        = _pcfg1.get('ye_key')

    for _sample1 in _samples1:
        _chip1 = data_dict.get(_sample1, {})
        _excl_raw1 = _pcfg1.get('exclude_resonators')
        if isinstance(_excl_raw1, dict):
            _excl1 = _excl_raw1.get(_sample1, []) or []
        else:
            _excl1 = _excl_raw1 or []
        _res_keys1 = [
            k for k, v in _chip1.items()
            if isinstance(v, dict)
            and (_res_filter1 is None or k in _res_filter1)
            and k not in _excl1
        ]
        _color1, _marker1 = get_sample_style(_sample1)

        for _rk1 in _res_keys1:
            _rd1 = _chip1[_rk1]
            if _xk1 not in _rd1 or _yk1 not in _rd1:
                continue
            _x1  = _rd1[_xk1]
            _y1  = _rd1[_yk1]
            _ye1 = _rd1.get(_yek1) if _yek1 else None
            # Normalise Qi data to millions (y / 1e6) so axis reads in units of 10^6
            if 'Qi' in _yk1:
                _y1 = _y1 / 1e6
                if _ye1 is not None:
                    _ye1 = _ye1 / 1e6
            # Normalise frequency data to the first index (f / f_0)
            if 'Freq' in _yk1:
                _f01 = float(_y1[0]) if hasattr(_y1, '__len__') and len(_y1) > 0 else 1.0
                if _f01 != 0:
                    _y1  = _y1  / _f01
                    if _ye1 is not None:
                        _ye1 = _ye1 / _f01
            _ax1.errorbar(
                _x1, _y1, yerr=_ye1,
                fmt=_marker1 + '-',
                ms=FMT['marker_size'],
                capsize=FMT['cap_size'],
                elinewidth=FMT['err_lw'],
                color=_color1,
            )

    # ── Per-panel cosmetics ───────────────────────────────────────────────
    apply_axes_style(
        _ax1,
        xlabel=_pcfg1.get('xlabel', ''),
        ylabel=_pcfg1.get('ylabel', ''),
        xscale=_pcfg1.get('xscale', 'linear'),
        yscale=_pcfg1.get('yscale', 'linear'),
    )
    _xlim1 = _pcfg1.get('xlim')
    _ylim1 = _pcfg1.get('ylim')
    if _xlim1: _ax1.set_xlim(*_xlim1)
    if _ylim1: _ax1.set_ylim(*_ylim1)
    _ptitle1 = _pcfg1.get('title')
    if _ptitle1: _ax1.set_title(_ptitle1, fontsize=FMT['title_size'] * 0.6)

# ── Figure-level legend ───────────────────────────────────────────────────
if FIG1_LEGEND:
    _leg1 = fig1.legend(
        handles=make_legend_handles(FIG1_LEGEND),
        loc='center left',
        bbox_to_anchor=(FIG1_LEGEND_X, FIG1_LEGEND_Y),
        bbox_transform=fig1.transFigure,
        fontsize=FMT['legend_size'],
        frameon=True,
        borderpad=0.6,
        handlelength=1.4,
        handletextpad=0.5,
        labelspacing=0.4,
    )

# Panel label on first axes
_axes_flat1[0].text(
    0.02, 0.97, FIG1_LABEL,
    transform=_axes_flat1[0].transAxes,
    fontsize=FMT['title_size'], fontweight='bold', va='top', ha='left',
)

save_figure(fig1, FIG1_FILENAME)


/var/folders/33/7p78xkb10gs52m9wzf0h3ls40000gn/T/ipykernel_29330/3354687578.py:248: UserWarning: AutoMinorLocator does not work with logarithmic scale
  fig.tight_layout()   # reflow layout before saving
/var/folders/33/7p78xkb10gs52m9wzf0h3ls40000gn/T/ipykernel_29330/3354687578.py:248: UserWarning: The figure layout has changed to tight
  fig.tight_layout()   # reflow layout before saving
/var/folders/33/7p78xkb10gs52m9wzf0h3ls40000gn/T/ipykernel_29330/3354687578.py:252: UserWarning: AutoMinorLocator does not work with logarithmic scale
  pdf.savefig(fig, bbox_inches='tight', dpi=dpi)


  Saved → paper_figures/fig1.pdf  (86.5 KB)


/var/folders/33/7p78xkb10gs52m9wzf0h3ls40000gn/T/ipykernel_29330/3354687578.py:254: UserWarning: AutoMinorLocator does not work with logarithmic scale
  plt.show()   # displays in the external Python figure viewer


PosixPath('paper_figures/fig1.pdf')

### Figure Cell 2 (Old Figure 2)
*Frequency error vs temperature*  
Edit `FIG2_*` variables in the CONFIG block to customise.

In [7]:
data_dict.keys()

dict_keys(['Sample A', 'Sample B', 'Sample C', 'Sample D', 'Sample E', 'Sample N'])

In [32]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  FIGURE CELL 2 — Figure 2 Title                                       ║
# ╚══════════════════════════════════════════════════════════════════════╝
# Requires: Cell 1 (data_dict) and Cell 2 (formatting helpers).
# Edit the CONFIG block below; do not change code below '# ── END CONFIG'.

# ── CONFIG ────────────────────────────────────────────────────────────────

FIG2_TITLE    = '$Q_i$ and Frequency vs Temperature'
FIG2_LABEL    = '(b)'        # panel label drawn in top-left corner of panel 0
FIG2_FILENAME = 'fig2'               # saved to ./paper_figures/

# Figure dimensions in inches (width, height). PHYSICAL size of the exported PDF.
FIG2_WIDTH  = 10   # inches
FIG2_HEIGHT = 4    # inches

# Grid layout: set rows × cols explicitly.
# Unused trailing slots are hidden automatically.
FIG2_ROWS = 1   # number of subplot rows
FIG2_COLS = 2   # number of subplot columns

# ── Figure-level legend ───────────────────────────────────────────────────
# One legend for the whole figure, drawn outside the panels.
# Each entry is a tuple: (label_string, color, marker_shape)
#
# marker_shape uses standard matplotlib marker codes, e.g.:
#   'o' circle  's' square  '^' triangle-up  'D' diamond
#   'v' triangle-down  '>' triangle-right  'p' pentagon  '*' star
#
# Set FIG2_LEGEND = [] to suppress the legend entirely.

# List sample names to include in the legend (order = display order).
FIG2_LEGEND = [
    'Sample A',
    'Sample B',
    'Sample C',
    'Sample D',
    'Sample E',
]
# Legend anchor position in figure-fraction coordinates (0–1).
# (1.0, 0.5) = right edge, vertically centred.  Increase X beyond 1 to
# push further right if panels are wide; decrease to move left into whitespace.
FIG2_LEGEND_X = 0.91  # horizontal anchor (figure fraction); sits in the right-margin gap
FIG2_LEGEND_Y = 0.5   # vertical anchor   (figure fraction)

# ── Per-panel configuration ───────────────────────────────────────────────
# FIG2_PANELS is a list of dicts, one per panel (left-to-right, top-to-bottom).
# Panels beyond ROWS × COLS are ignored. Unfilled grid slots are hidden.
#
# Required keys per panel dict:
#   'samples'            : list of sample names from data_dict to plot in this panel.
#                          Use None or [] to leave the panel blank.
#   'resonators'         : list of resonator keys to include, e.g. ['Res1', 'Res2'].
#                          Use None to plot ALL resonators found in each sample.
#   'exclude_resonators' : list of resonator keys to skip, e.g. ['Res1'].
#                          Applied after 'resonators'; use None to skip nothing.
#                          Per-sample exclusions: dict keyed by sample name, e.g.
#                          {'Sample A': ['Res1'], 'Sample B': ['Res3']}
#   'x_key'              : data_dict key for the x axis, e.g. 'Num_Photon' or 'Temp'.
#   'y_key'              : data_dict key for the y axis, e.g. 'Qi_Power' or 'Qi_Temp'.
#   'ye_key'             : data_dict key for y error bars (set to None to suppress).
#
# Optional keys per panel dict:
#   'xlabel'     : x-axis label string  (default: '')
#   'ylabel'     : y-axis label string  (default: '')
#   'xscale'     : 'linear' or 'log'   (default: 'linear')
#   'yscale'     : 'linear' or 'log'   (default: 'linear')
#   'xlim'       : (lo, hi) or None    (default: None = auto)
#   'ylim'       : (lo, hi) or None    (default: None = auto)
#   'title'      : panel title string  (default: None = no title)
#
# Common x/y key names:
#   Qi_Temp, Qi_Power, Num_Photon, Temp, Freq,
#   Q_Temp, Qc_Temp, Freq_err_Temp, Power_Freq, Power_Freq_err

FIG2_PANELS = [
    # Panel 0 — Qi vs Temperature, Samples A-E, excluding Res1_NoTemp from Sample A
    dict(
        samples             = ['Sample A', 'Sample B', 'Sample C', 'Sample D', 'Sample E'],
        resonators          = None,
        exclude_resonators  = {'Sample A': ['Res1_NoTemp']},
        x_key               = 'Temp',
        y_key               = 'Qi_Temp',
        ye_key              = 'Qi_Temp_err',
        xlabel              = 'T (mK)',
        ylabel              = r'$Q_i$ $(10^6)$',
        xscale              = 'linear',
        yscale              = 'linear',
        xlim                = None,
        ylim                = None,
        title               = r'$Q_i$ vs Temperature',
    ),
    # Panel 1 — Frequency vs Temperature, Samples A-E, same exclusions
    dict(
        samples             = ['Sample A', 'Sample B', 'Sample C', 'Sample D', 'Sample E'],
        resonators          = None,
        exclude_resonators  = {'Sample A': ['Res1_NoTemp']},
        x_key               = 'Temp',
        y_key               = 'Freq',
        ye_key              = None,
        xlabel              = 'T (mK)',
        ylabel              = 'Frequency (Hz)',
        xscale              = 'linear',
        yscale              = 'linear',
        xlim                = None,
        ylim                = None,
        title               = 'Frequency vs Temperature',
    ),
]

# ── END CONFIG ────────────────────────────────────────────────────────────

# ── Build figure ──────────────────────────────────────────────────────────
_n_slots2 = FIG2_ROWS * FIG2_COLS
_n_panels2 = min(len(FIG2_PANELS), _n_slots2)

_fig2_size = (FIG2_WIDTH, FIG2_HEIGHT)
if FIG2_ROWS == 1 and FIG2_COLS == 1:
    fig2, _ax_raw2 = plt.subplots(figsize=_fig2_size,
                                      layout='constrained')
    _axes_flat2 = np.array([_ax_raw2])
else:
    fig2, _ax_raw2 = plt.subplots(FIG2_ROWS, FIG2_COLS, figsize=_fig2_size,
                                      squeeze=False, layout='constrained')
    _axes_flat2 = np.array(_ax_raw2).flatten()

# layout='constrained' (set above) manages margins automatically.
# subplots_adjust is not used — it conflicts with constrained_layout.
# To add extra padding around the figure, edit FMT['left'/'right'/'bottom'/'top']
# then call fig2.set_constrained_layout_pads(w_pad=..., h_pad=...) after this block.

# Hide unused grid slots
for _k2 in range(_n_panels2, len(_axes_flat2)):
    _axes_flat2[_k2].axis('off')

# ── Plotting loop (no per-panel legend) ───────────────────────────────────
for _pi2, _pcfg2 in enumerate(FIG2_PANELS[:_n_panels2]):
    _ax2 = _axes_flat2[_pi2]

    _samples2    = _pcfg2.get('samples') or []
    _res_filter2 = _pcfg2.get('resonators')
    _xk2         = _pcfg2['x_key']
    _yk2         = _pcfg2['y_key']
    _yek2        = _pcfg2.get('ye_key')

    for _sample2 in _samples2:
        _chip2 = data_dict.get(_sample2, {})
        _excl_raw2 = _pcfg2.get('exclude_resonators')
        if isinstance(_excl_raw2, dict):
            _excl2 = _excl_raw2.get(_sample2, []) or []
        else:
            _excl2 = _excl_raw2 or []
        _res_keys2 = [
            k for k, v in _chip2.items()
            if isinstance(v, dict)
            and (_res_filter2 is None or k in _res_filter2)
            and k not in _excl2
        ]
        _color2, _marker2 = get_sample_style(_sample2)

        for _rk2 in _res_keys2:
            _rd2 = _chip2[_rk2]
            if _xk2 not in _rd2 or _yk2 not in _rd2:
                continue
            _x2  = _rd2[_xk2]
            _y2  = _rd2[_yk2]
            _ye2 = _rd2.get(_yek2) if _yek2 else None
            # Normalise Qi data to millions (y / 1e6) so axis reads in units of 10^6
            if 'Qi' in _yk2:
                _y2 = _y2 / 1e6
                if _ye2 is not None:
                    _ye2 = _ye2 / 1e6
            # Normalise frequency data to the first index (f / f_0)
            if 'Freq' in _yk2:
                _f02 = float(_y2[0]) if hasattr(_y2, '__len__') and len(_y2) > 0 else 1.0
                if _f02 != 0:
                    _y2  = _y2  / _f02
                    if _ye2 is not None:
                        _ye2 = _ye2 / _f02
            _ax2.errorbar(
                _x2, _y2, yerr=_ye2,
                fmt=_marker2 + '-',
                ms=FMT['marker_size'],
                capsize=FMT['cap_size'],
                elinewidth=FMT['err_lw'],
                color=_color2,
            )

    # ── Per-panel cosmetics ───────────────────────────────────────────────
    apply_axes_style(
        _ax2,
        xlabel=_pcfg2.get('xlabel', ''),
        ylabel=_pcfg2.get('ylabel', ''),
        xscale=_pcfg2.get('xscale', 'linear'),
        yscale=_pcfg2.get('yscale', 'linear'),
    )
    _xlim2 = _pcfg2.get('xlim')
    _ylim2 = _pcfg2.get('ylim')
    if _xlim2: _ax2.set_xlim(*_xlim2)
    if _ylim2: _ax2.set_ylim(*_ylim2)
    _ptitle2 = _pcfg2.get('title')
    if _ptitle2: _ax2.set_title(_ptitle2, fontsize=FMT['title_size'] * 0.6)

# ── Figure-level legend ───────────────────────────────────────────────────
if FIG2_LEGEND:
    _leg2 = fig2.legend(
        handles=make_legend_handles(FIG2_LEGEND),
        loc='center left',
        bbox_to_anchor=(FIG2_LEGEND_X, FIG2_LEGEND_Y),
        bbox_transform=fig2.transFigure,
        fontsize=FMT['legend_size'],
        frameon=True,
        borderpad=0.6,
        handlelength=1.4,
        handletextpad=0.5,
        labelspacing=0.4,
    )

# Panel label on first axes
_axes_flat2[0].text(
    0.02, 0.97, FIG2_LABEL,
    transform=_axes_flat2[0].transAxes,
    fontsize=FMT['title_size'], fontweight='bold', va='top', ha='left',
)

save_figure(fig2, FIG2_FILENAME)


/var/folders/33/7p78xkb10gs52m9wzf0h3ls40000gn/T/ipykernel_29330/3354687578.py:248: UserWarning: The figure layout has changed to tight
  fig.tight_layout()   # reflow layout before saving


  Saved → paper_figures/fig2.pdf  (74.8 KB)


PosixPath('paper_figures/fig2.pdf')

### Figure Cell 3 (Figure S4)

In [18]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  PHOTON NUMBER HELPER                                                ║
# ║  Calculates the intra-resonator photon number from temp sweep data.  ║
# ║  Run once after Cell 2. Used by Figure Cell 6.                       ║
# ╚══════════════════════════════════════════════════════════════════════╝

# Fixed experimental constants
_PWR_DBM  = -10 - 96   # drive power at device: -10 dBm source - 96 dB attenuation
_GAMMA    = 0.0        # impedance mismatch (≈0 for these devices)
_HALFWAVE = 0          # quarter-wave resonators


def PhotonNumber(pwr_dB, f0, Q, Qc, gamma=_GAMMA, halfwave=_HALFWAVE):
    """
    Calculate the intra-resonator photon number.

    Parameters
    ----------
    pwr_dB   : float        Input power at device in dBm
    f0       : float/array  Resonant frequency in Hz
    Q        : float/array  Total quality factor
    Qc       : float/array  Coupling (external) quality factor
    gamma    : float        Impedance mismatch parameter (default 0.0)
    halfwave : int          0 = quarter-wave, 1 = half-wave (default 0)

    Returns
    -------
    N : array  Intra-resonator photon number
    """
    pwr_W    = 10**((float(pwr_dB) - 30) / 10)
    factor_P = -halfwave + 2       # = 2 for quarter-wave
    factor_N = -2 * halfwave + 4   # = 4 for quarter-wave
    P_int    = (1 - gamma**2) * (factor_P * np.asarray(Q)**2 / (np.pi * np.asarray(Qc))) * pwr_W
    N        = P_int / (factor_N * 6.626e-34 * np.asarray(f0)**2)
    return N


def compute_photon_vs_temp(chip, res_key):
    """
    Compute intra-resonator photon number at each temperature point.

    Reads from the resonator dict:
        Temp    — sample temperature during the sweep
        Qi_Temp — internal quality factor vs temperature
        Qc_Temp — coupling quality factor vs temperature
        Freq    — resonant frequency vs temperature

    Drive power is fixed at _PWR_DBM = -10 - 96 dBm.
    Total Q is derived from Qi and Qc via Matthiessen's rule.

    Returns (Temp, N_photon) arrays, or (None, None) if any key is missing.
    """
    rd = chip.get(res_key, {})
    needed = ['Temp', 'Qi_Temp', 'Qc_Temp', 'Freq']
    if not all(k in rd for k in needed):
        return None, None

    Temp = np.asarray(rd['Temp'],    dtype=float)
    Qi   = np.asarray(rd['Qi_Temp'], dtype=float)
    Qc   = np.asarray(rd['Qc_Temp'], dtype=float)
    freq = np.asarray(rd['Freq'],    dtype=float)

    # Total Q from Matthiessen's rule: 1/Q = 1/Qi + 1/Qc
    Q = 1.0 / (1.0/Qi + 1.0/Qc)

    # f0: per-point if shapes match Temp, else use first value as scalar
    f0 = freq if freq.shape == Temp.shape else float(freq.flat[0])

    N = PhotonNumber(_PWR_DBM, f0, Q, Qc)
    return Temp, N


print(f"PhotonNumber helper ready.")
print(f"  Fixed constants: power = {_PWR_DBM} dBm, gamma = {_GAMMA}, halfwave = {_HALFWAVE} (quarter-wave)")
print(f"  compute_photon_vs_temp(chip, res_key) → (Temp, N)")

PhotonNumber helper ready.
  Fixed constants: power = -106 dBm, gamma = 0.0, halfwave = 0 (quarter-wave)
  compute_photon_vs_temp(chip, res_key) → (Temp, N)


In [20]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  FIGURE CELL 6 — Photon Number vs Temperature                        ║
# ╚══════════════════════════════════════════════════════════════════════╝
# Requires: Cell 1 (data_dict), Cell 2 (formatting helpers),
#           PhotonNumber helper cell.
# Edit the CONFIG block below; do not change code below '# ── END CONFIG'.

# ── CONFIG ────────────────────────────────────────────────────────────────

FIG6_FILENAME = 'fig6_photon_vs_temp'   # saved to ./paper_figures/
FIG6_LABEL    = '(f)'

# Figure dimensions
FIG6_WIDTH  = 10   # inches
FIG6_HEIGHT = 4    # inches

# Grid: 1 row × 2 cols
#   Panel 0 — all resonators per sample (individual lines)
#   Panel 1 — one averaged line per sample
FIG6_ROWS = 1
FIG6_COLS = 2

# Samples to include (must have Temp + Qi_Temp + Qc_Temp + Freq)
FIG6_SAMPLES = ['Sample A', 'Sample B', 'Sample C', 'Sample D', 'Sample E']

# Resonators to exclude per sample (same dict syntax as other figure cells)
FIG6_EXCLUDE = {'Sample A': ['Res1_NoTemp']}

# Axis config (shared by both panels)
FIG6_XLABEL = 'T (mK)'
FIG6_YLABEL = r'$\langle n \rangle$'
FIG6_XSCALE = 'linear'
FIG6_YSCALE = 'log'
FIG6_XLIM   = None
FIG6_YLIM   = None

# Legend
FIG6_LEGEND   = [s for s in FIG6_SAMPLES]
FIG6_LEGEND_X = 0.91
FIG6_LEGEND_Y = 0.5

# ── END CONFIG ────────────────────────────────────────────────────────────

fig6, _ax_raw6 = plt.subplots(FIG6_ROWS, FIG6_COLS, figsize=(FIG6_WIDTH, FIG6_HEIGHT),
                               squeeze=False, layout='constrained')
_ax6_all = _ax_raw6[0, 0]   # Panel 0: all resonators
_ax6_avg = _ax_raw6[0, 1]   # Panel 1: per-sample average

# ── Compute and store photon arrays, then plot ────────────────────────────
# Also print arrays in curated_data format
print("# ── Photon number vs temperature ─────────────────────────────────────")

for _sample6 in FIG6_SAMPLES:
    _chip6  = data_dict.get(_sample6, {})
    _excl6  = FIG6_EXCLUDE.get(_sample6, [])
    _color6, _marker6 = get_sample_style(_sample6)

    _res_keys6 = [
        k for k, v in _chip6.items()
        if isinstance(v, dict) and k not in _excl6
    ]

    _all_T6 = []
    _all_N6 = []

    for _rk6 in _res_keys6:
        _T6, _N6 = compute_photon_vs_temp(_chip6, _rk6)
        if _T6 is None:
            continue

        # Panel 0: individual resonator line
        _ax6_all.plot(
            _T6, _N6,
            marker=_marker6, ms=FMT['marker_size'],
            color=_color6, linestyle='-',
            linewidth=FMT['line_width'],
        )
        _all_T6.append(_T6)
        _all_N6.append(_N6)

        # Print in curated_data format
        _varname6 = f"{_sample6.replace(' ', '_')}_{_rk6}_N_photon"
        _arr_str6 = np.array2string(_N6, separator=', ',
                                    formatter={'float_kind': lambda x: f'{x:.6e}'})
        print(f"{_varname6} = np.array({_arr_str6})")

    # Panel 1: average across resonators with standard error of the mean
    if _all_N6:
        _T_common6 = _all_T6[0]
        _N_stack6  = []
        for _Ti6, _Ni6 in zip(_all_T6, _all_N6):
            if np.array_equal(_Ti6, _T_common6):
                _N_stack6.append(_Ni6)
            else:
                _N_stack6.append(np.interp(_T_common6, _Ti6, _Ni6))

        _N_arr6  = np.array(_N_stack6)
        _N_mean6 = np.mean(_N_arr6, axis=0)
        # Standard error of the mean = std / sqrt(n_resonators)
        # This is the uncertainty on the sample mean, not the spread of resonators
        _n_res6  = len(_N_stack6)
        _N_sem6  = np.std(_N_arr6, axis=0, ddof=1) / np.sqrt(_n_res6) if _n_res6 > 1 else None

        _ax6_avg.errorbar(
            _T_common6, _N_mean6, yerr=_N_sem6,
            fmt=_marker6 + '-',
            ms=FMT['marker_size'],
            capsize=FMT['cap_size'],
            elinewidth=FMT['err_lw'],
            color=_color6,
        )

        # Print mean and SEM arrays
        _mean_var6 = f"{_sample6.replace(' ', '_')}_N_photon_mean"
        _sem_var6  = f"{_sample6.replace(' ', '_')}_N_photon_sem"
        _mean_str6 = np.array2string(_N_mean6, separator=', ',
                                     formatter={'float_kind': lambda x: f'{x:.6e}'})
        print(f"{_mean_var6} = np.array({_mean_str6})")
        if _N_sem6 is not None:
            _sem_str6 = np.array2string(_N_sem6, separator=', ',
                                        formatter={'float_kind': lambda x: f'{x:.6e}'})
            print(f"{_sem_var6}  = np.array({_sem_str6})")

print("# ─────────────────────────────────────────────────────────────────────")

# ── Cosmetics ─────────────────────────────────────────────────────────────
for _ax6 in [_ax6_all, _ax6_avg]:
    apply_axes_style(_ax6,
                     xlabel=FIG6_XLABEL,
                     ylabel=FIG6_YLABEL,
                     xscale=FIG6_XSCALE,
                     yscale=FIG6_YSCALE)
    if FIG6_XLIM: _ax6.set_xlim(*FIG6_XLIM)
    if FIG6_YLIM: _ax6.set_ylim(*FIG6_YLIM)

_ax6_all.set_title('All Resonators',    fontsize=FMT['font_size'])
_ax6_avg.set_title('Per-Sample Average', fontsize=FMT['font_size'])

_axes_flat6 = _ax_raw6.flatten()
_axes_flat6[0].text(
    0.02, 0.97, FIG6_LABEL,
    transform=_axes_flat6[0].transAxes,
    fontsize=FMT['title_size'], fontweight='bold', va='top', ha='left',
)

if FIG6_LEGEND:
    fig6.legend(
        handles=make_legend_handles(FIG6_LEGEND),
        loc='center left',
        bbox_to_anchor=(FIG6_LEGEND_X, FIG6_LEGEND_Y),
        bbox_transform=fig6.transFigure,
        fontsize=FMT['legend_size'],
        frameon=True,
        borderpad=0.6,
        handlelength=1.4,
        handletextpad=0.5,
        labelspacing=0.4,
    )

save_figure(fig6, FIG6_FILENAME)

# ── Photon number vs temperature ─────────────────────────────────────
Sample_A_Res2_N_photon = np.array([1.508825e+03, 1.418251e+03, 1.341185e+03, 1.289955e+03, 1.263892e+03,
 1.234007e+03, 1.171214e+03, 1.044452e+03, 7.891492e+02, 6.441431e+02,
 5.355419e+02, 4.172398e+02, 3.350940e+02, 2.906462e+02, 2.270303e+02,
 1.617853e+02, 8.436607e+01, 5.362183e+01, 3.354164e+01, 2.009970e+01,
 1.319582e+01, 7.409831e+00])
Sample_A_Res3_N_photon = np.array([1.778722e+03, 1.490371e+03, 1.406797e+03, 1.324909e+03, 1.253091e+03,
 1.177250e+03, 1.156810e+03, 1.008196e+03, 8.702384e+02, 7.411560e+02,
 6.124579e+02, 4.789313e+02, 3.879973e+02, 3.696750e+02, 2.816429e+02,
 2.125971e+02, 1.104983e+02, 7.294971e+01, 4.714334e+01, 2.952486e+01,
 1.637407e+01, 7.789054e+00])
Sample_A_Res4_N_photon = np.array([4.926663e+02, 3.663830e+02, 4.364763e+02, 4.025573e+02, 3.953146e+02,
 3.726869e+02, 4.610867e+02, 3.703334e+02, 3.572201e+02, 2.821703e+02,
 2.865988e+02, 2.291502e+02, 1.925181e+02, 1.535437e+02,

/var/folders/33/7p78xkb10gs52m9wzf0h3ls40000gn/T/ipykernel_29330/3354687578.py:248: UserWarning: AutoMinorLocator does not work with logarithmic scale
  fig.tight_layout()   # reflow layout before saving
/var/folders/33/7p78xkb10gs52m9wzf0h3ls40000gn/T/ipykernel_29330/3354687578.py:248: UserWarning: The figure layout has changed to tight
  fig.tight_layout()   # reflow layout before saving
/var/folders/33/7p78xkb10gs52m9wzf0h3ls40000gn/T/ipykernel_29330/3354687578.py:252: UserWarning: AutoMinorLocator does not work with logarithmic scale
  pdf.savefig(fig, bbox_inches='tight', dpi=dpi)


  Saved → paper_figures/fig6_photon_vs_temp.pdf  (46.7 KB)


/var/folders/33/7p78xkb10gs52m9wzf0h3ls40000gn/T/ipykernel_29330/3354687578.py:254: UserWarning: AutoMinorLocator does not work with logarithmic scale
  plt.show()   # displays in the external Python figure viewer


PosixPath('paper_figures/fig6_photon_vs_temp.pdf')

/opt/miniconda3/lib/python3.8/site-packages/ipykernel/eventloops.py:145: UserWarning: AutoMinorLocator does not work with logarithmic scale
  el.exec() if hasattr(el, "exec") else el.exec_()
